# Kaleb Alejandro Aguilar Ávila
## Tarea 1 — Parte 2: Machine Translation (inglés → español)
### Seq2Seq recurrente con atención Q, K, V (mLLMs, CIMAT — Otoño 2026)

Datos: Tatoeba (`data/tatoeba_train.tsv` 80%, `data/tatoeba_val.tsv` 10%, `data/tatoeba_test.tsv` 10%).

- **Train** → entrenamiento (subconjuntos anidados de X y 2X pares).
- **Val** → early stopping / decisiones de hiperparámetros.
- **Test** → sólo evaluación final (BLEU y chrF). **Nunca** se usa para tomar decisiones.

Código: `data.py` (datos, tokenización, vocabularios) y `seq2seq_attention.py` (modelo, entrenamiento, traducción y métricas).

**LLM utilizado para Vibe Coding:** _(completar)_

## 0. Imports y configuración

In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import data as D
import seq2seq_attention as S

SEED = 1111
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
os.makedirs('checkpoints', exist_ok=True)

## 1. Datos

In [ ]:
train_pairs = D.load_tatoeba('data/tatoeba_train.tsv')
val_pairs   = D.load_tatoeba('data/tatoeba_val.tsv')
test_pairs  = D.load_tatoeba('data/tatoeba_test.tsv')
print(f'|Train|={len(train_pairs)} | |Val|={len(val_pairs)} | |Test|={len(test_pairs)}')

lens_en = [len(D.tokenize(en)) for en, _ in train_pairs]
lens_es = [len(D.tokenize(es)) for _, es in train_pairs]
print(f'Longitud media (tokens) en={np.mean(lens_en):.1f} es={np.mean(lens_es):.1f} | p99 en={np.percentile(lens_en, 99):.0f} es={np.percentile(lens_es, 99):.0f}')
train_pairs[:5]

## 2. Configuración común a los dos experimentos

Ambos experimentos usan **exactamente la misma arquitectura y configuración de entrenamiento**; sólo cambia el número de pares de Train (X vs 2X). Los X pares del experimento 1 son un **subconjunto** de los 2X del experimento 2.

Ajusta `X` según tus recursos de cómputo (debe cumplirse 2X ≤ |Train|).

In [ ]:
X = 50_000   # <- ajustar según recursos (2X = 100,000 <= |Train|)

DATA_CFG  = dict(lower=True, min_freq=2, max_len_train=30, batch_size=128)
MODEL_CFG = dict(emb_dim=256, hidden_dim=512, num_layers=2, attn_dim=256, n_heads=4, dropout=0.3)
TRAIN_CFG = dict(lr=1e-3, num_epochs=20, patience=3, clip=1.0, teacher_forcing=1.0, label_smoothing=0.1)

subset_x, subset_2x = D.make_nested_subsets(train_pairs, X, seed=SEED)
assert set(subset_x) <= set(subset_2x)
print(f'Experimento 1: {len(subset_x)} pares | Experimento 2: {len(subset_2x)} pares')

In [ ]:
def run_experiment(name, pairs):
    """Construye vocabularios con `pairs`, entrena con early stopping sobre Val y regresa todo lo necesario."""
    set_seed(SEED)
    src_vocab, tgt_vocab = D.build_vocabs(pairs, min_freq=DATA_CFG['min_freq'], lower=DATA_CFG['lower'])
    print(f'[{name}] |V_en|={len(src_vocab)} |V_es|={len(tgt_vocab)}')

    train_loader = D.make_loader(pairs, src_vocab, tgt_vocab, DATA_CFG['batch_size'], shuffle=True,
                                 lower=DATA_CFG['lower'], max_len=DATA_CFG['max_len_train'])
    val_loader = D.make_loader(val_pairs, src_vocab, tgt_vocab, DATA_CFG['batch_size'],
                               lower=DATA_CFG['lower'])

    model = S.Seq2SeqAttention(len(src_vocab), len(tgt_vocab), src_pad_id=src_vocab.pad_id,
                               tgt_pad_id=tgt_vocab.pad_id, **MODEL_CFG).to(device)
    print(f'[{name}] parámetros: {S.count_parameters(model):,}')

    # BLEU sobre una muestra fija de Val para monitorear (NO Test)
    val_sample = val_pairs[:1000]
    def val_bleu():
        hyps = S.translate(model, [en for en, _ in val_sample], src_vocab, tgt_vocab, device,
                           lower=DATA_CFG['lower'])
        return S.compute_metrics(hyps, [es for _, es in val_sample], lowercase=DATA_CFG['lower'])['BLEU']

    t0 = time.time()
    history = S.fit(model, train_loader, val_loader, tgt_vocab.pad_id, device,
                    save_path=f'checkpoints/seq2seq_{name}.pt', val_bleu_fn=val_bleu, **TRAIN_CFG)
    train_time = time.time() - t0
    print(f'[{name}] tiempo de entrenamiento: {train_time/60:.1f} min')
    return dict(model=model, src_vocab=src_vocab, tgt_vocab=tgt_vocab, history=history, train_time=train_time)

## 3. (Punto 1) Modelo

- **Encoder:** Embedding → GRU bidireccional de 2 capas → salidas `H` (B, S, 2h).
- **Decoder:** GRU de 2 capas con *input feeding*; en cada paso `t`:
  - **Query**: `Q = s_t · W_Q` (estado oculto del decoder)
  - **Keys**: `K = H · W_K` (salidas del encoder)
  - **Values**: `V = H · W_V` (salidas del encoder)
  - **Pesos de atención**: `α_t = softmax(Q Kᵀ / √d_k)` con máscara sobre `<pad>` (4 cabezas)
  - **Contexto**: `c_t = α_t · V`, luego `z_t = tanh(W_c [s_t; c_t])` y `softmax(W_o z_t)`.
- Ver la clase `QKVAttention` en `seq2seq_attention.py`.

In [ ]:
import inspect
print(inspect.getsource(S.QKVAttention))

## 4. (Punto 2) Experimento 1 — X pares

In [ ]:
exp_x = run_experiment('X', subset_x)

## 5. (Punto 2) Experimento 2 — 2X pares

In [ ]:
exp_2x = run_experiment('2X', subset_2x)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, exp in [('X', exp_x), ('2X', exp_2x)]:
    h = exp['history']
    ep = range(1, len(h['val_loss']) + 1)
    ax[0].plot(ep, h['train_loss'], '--', label=f'train {name}')
    ax[0].plot(ep, h['val_loss'], marker='o', label=f'val {name}')
    ax[1].plot(ep, h['val_bleu'], marker='o', label=f'{name}')
ax[0].set_title('Loss'); ax[0].set_xlabel('Época'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('BLEU (muestra de Val)'); ax[1].set_xlabel('Época'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. (Punto 3) Evaluación final en Test — BLEU y chrF

In [ ]:
test_src = [en for en, _ in test_pairs]
test_ref = [es for _, es in test_pairs]

rows = []
for name, exp, n in [('X', exp_x, X), ('2X', exp_2x, 2 * X)]:
    t0 = time.time()
    exp['test_hyps'] = S.translate(exp['model'], test_src, exp['src_vocab'], exp['tgt_vocab'], device,
                                   lower=DATA_CFG['lower'])
    m = S.compute_metrics(exp['test_hyps'], test_ref, lowercase=DATA_CFG['lower'])
    print(name, m['bleu_obj'], '|', m['chrf_obj'], f'| {time.time()-t0:.0f}s')
    rows.append({'Experimento': name, 'Pares de Train': n, 'Épocas': len(exp['history']['val_loss']),
                 'Val loss': round(exp['history']['best_val_loss'], 3),
                 'BLEU (Test)': round(m['BLEU'], 2), 'chrF (Test)': round(m['chrF'], 2),
                 'Tiempo (min)': round(exp['train_time'] / 60, 1)})

pd.DataFrame(rows)

### 6.1 Ejemplos de Test (≥ 10)

In [ ]:
rng = random.Random(SEED)
idx = rng.sample(range(len(test_pairs)), 12)
examples = pd.DataFrame({
    'Inglés (original)': [test_src[i] for i in idx],
    'Referencia (es)': [test_ref[i] for i in idx],
    'Modelo X': [exp_x['test_hyps'][i] for i in idx],
    'Modelo 2X': [exp_2x['test_hyps'][i] for i in idx],
})
with pd.option_context('display.max_colwidth', None):
    display(examples)

### 6.2 Mapa de atención (ejemplo)

In [ ]:
sentence = test_src[idx[0]]
src_toks, out_toks, A = S.translate_with_attention(exp_2x['model'], sentence, exp_2x['src_vocab'],
                                                   exp_2x['tgt_vocab'], device)
plt.figure(figsize=(6, 5))
plt.imshow(A.numpy(), cmap='viridis', aspect='auto')
plt.xticks(range(len(src_toks)), src_toks, rotation=90); plt.yticks(range(len(out_toks)), out_toks)
plt.xlabel('Inglés (Keys/Values)'); plt.ylabel('Español generado (Queries)'); plt.colorbar()
plt.title('Pesos de atención (promedio de cabezas) — modelo 2X'); plt.show()

## 7. Comentarios

**Resultados X vs 2X:** _(completar)_

**Errores / diferencias observadas:** _(completar)_

**LLM usado, decisiones, dificultades, alucinaciones, recursos de cómputo y tiempo de entrenamiento:** _(completar)_